# 01 · Exploratory Data Analysis

Before committing to a final question, I want to understand who is in this dataset, what the Combine measures, and what passing plays look like. Open questions going in:

- Who are the 510 prospects: positions, draft rounds, colleges?
- How do Combine results differ by position?
- How far are passes thrown, and which teams throw deepest?
- How long does the quarterback hold the ball before throwing?
- Which defensive backs play the most, and in which coverages?
- Do Combine numbers relate to who becomes a starter?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

CANDIDATES = [Path("/kaggle/input/nfl-big-data-bowl-2027"), Path("../data"), Path("data")]
DATA = next(p for p in CANDIDATES if (p / "players.csv").exists())
print("Reading from", DATA.resolve())

DB_POSITIONS = {"CB", "SS", "FS", "DB"}

In [ ]:
players  = pd.read_csv(DATA / "players.csv")
combine  = pd.read_csv(DATA / "combine_results.csv")
careers  = pd.read_csv(DATA / "player_career_successes.csv")
games    = pd.read_csv(DATA / "games.csv")
pp       = pd.read_csv(DATA / "player_play.csv", low_memory=False)

for name, df in {"players": players, "combine": combine, "careers": careers, "games": games, "player_play": pp}.items():
    print(f"{name:12s} {df.shape[0]:>9,} rows x {df.shape[1]:>3} cols")

## 1. Who is in the cohort?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
players["nfl_position"].value_counts().plot.bar(ax=axes[0], title="Players by NFL position")
(players["draft_round"].fillna(0).astype(int).replace(0, "UDFA").astype(str)
    .value_counts().sort_index().plot.bar(ax=axes[1], title="Draft round (UDFA = undrafted)"))
plt.tight_layout()

In [ ]:
players["age_at_draft"] = (
    pd.to_datetime(players["draft_year"].astype(str) + "-04-25") - pd.to_datetime(players["birth_date"])
).dt.days / 365.25

display(players["college_conference"].value_counts().head(10).rename("players").to_frame())
display(players["college_name"].value_counts().head(10).rename("players").to_frame())
players["age_at_draft"].describe().round(2)

## 2. Combine results by position

In [ ]:
tests = ["forty", "ten_yd_split", "three_cone", "short_shuttle", "vertical", "broad_jump", "combine_weight"]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, tests):
    sns.boxplot(data=combine, x="combine_position", y=col, ax=ax)
    ax.set_title(col); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=45)
axes.flat[-1].axis("off")
plt.tight_layout()

combine[tests].isna().mean().round(3).rename("share_missing").to_frame()

## 3. Passing plays

`player_play` has one row per cohort player per play, so play-level fields repeat. For play-level questions I deduplicate on `(game_id, play_id)`.

In [ ]:
plays = (pp.drop_duplicates(["game_id", "play_id"])
           .query("pass_result.notna() and play_nullified_by_penalty != 'Y'", engine="python"))
passes = plays[plays["pass_result"].isin(["C", "I", "IN"])]
print(f"{len(passes):,} pass attempts")

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(passes["pass_length"], bins=60, ax=ax)
ax.set(title="Air yards per pass", xlabel="Air yards past the line of scrimmage")

In [ ]:
cols = ["game_id", "possession_team", "pass_length", "pass_result", "play_description"]
print("Longest passes:")
display(passes.nlargest(10, "pass_length")[cols])

by_team = (passes.groupby("possession_team")["pass_length"]
                 .agg(avg_air_yards="mean", deep_share=lambda s: (s >= 20).mean(), attempts="size")
                 .sort_values("avg_air_yards", ascending=False))
by_team.round(3).head(10)

In [ ]:
bins = [-10, 0, 10, 20, 70]
labels = ["behind LOS", "0-9", "10-19", "20+"]
passes["depth"] = pd.cut(passes["pass_length"], bins=bins, labels=labels)
(passes.groupby("depth", observed=True)["pass_result"]
       .value_counts(normalize=True).unstack().round(3))

## 4. Time before the throw

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(plays["dropback_duration"].dropna().clip(upper=8), bins=60, ax=ax)
ax.set(title="Time from snap to throw, sack, or pressure", xlabel="Seconds")

passes.groupby(pd.qcut(passes["dropback_duration"], 5), observed=True).agg(
    completion_rate=("pass_result", lambda s: (s == "C").mean()),
    int_rate=("pass_result", lambda s: (s == "IN").mean()),
    avg_air_yards=("pass_length", "mean"),
).round(3)

## 5. Defensive backs on the field

In [ ]:
db_ids = set(players.loc[players["nfl_position"].isin(DB_POSITIONS), "nfl_id"])
db_plays = pp[pp["nfl_id"].isin(db_ids)]
print(f"{len(db_ids)} defensive backs, {len(db_plays):,} player-plays")

top = (db_plays.groupby("nfl_id").size().rename("plays").to_frame()
               .join(players.set_index("nfl_id")[["display_name", "nfl_position", "draft_round"]])
               .sort_values("plays", ascending=False))
display(top.head(10))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
db_plays["team_coverage_man_zone"].value_counts().plot.bar(ax=axes[0], title="Team coverage: man vs zone")
db_plays["coverage_assignment"].value_counts().head(10).plot.bar(ax=axes[1], title="DB coverage assignments")
plt.tight_layout()

## 6. Combine numbers and becoming a starter

In [ ]:
df = (players.merge(combine, on=["nfl_id", "draft_year"], how="left")
             .merge(careers, on="nfl_id", how="left"))
df["start_rate"] = df["career_games_started"] / df["career_games_active"].replace(0, np.nan)
dbs = df[df["nfl_position"].isin(DB_POSITIONS)]

measures = ["forty", "ten_yd_split", "three_cone", "short_shuttle", "vertical", "broad_jump",
            "ngs_athleticism_score", "draft_overall_pick"]
dbs[measures + ["start_rate", "career_defensive_snaps"]].corr(method="spearman")[
    ["start_rate", "career_defensive_snaps"]].loc[measures].round(2)

## Observations

Notes on what stands out, recorded as the analysis runs. These feed the choice of final question in notebook 02.